In [ ]:
import os
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 20newsgroups_dataset.csv to 20newsgroups_dataset.csv


In [ ]:
df = pd.read_csv("20newsgroups_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (18846, 2)


In [ ]:
print("First 5 rows:")
print(df.head())

First 5 rows:
                                                text                  category
0  \n\nI am sure some bashers of Pens fans are pr...          rec.sport.hockey
1  My brother is in the market for a high-perform...  comp.sys.ibm.pc.hardware
2  \n\n\n\n\tFinally you said what you dream abou...     talk.politics.mideast
3  \nThink!\n\nIt's the SCSI card doing the DMA t...  comp.sys.ibm.pc.hardware
4  1)    I have an old Jasmine drive which I cann...     comp.sys.mac.hardware


In [ ]:
print("Dataset Information:")
print(df.info())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18846 entries, 0 to 18845
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      18466 non-null  object
 1   category  18846 non-null  object
dtypes: object(2)
memory usage: 294.6+ KB
None


In [ ]:
df.dropna(inplace=True)

df.drop_duplicates(inplace=True)

print(df.shape)

(18351, 2)


In [ ]:
selected_categories = [
    "comp.graphics",
    "rec.sport.hockey",
    "sci.space",
    "talk.politics.misc"
]

df = df[df["category"].isin(selected_categories)].copy()

print("Selected Categories:")
print(df["category"].value_counts())

print("\nTotal Samples:", len(df))

Selected Categories:
category
rec.sport.hockey      979
sci.space             959
comp.graphics         955
talk.politics.misc    760
Name: count, dtype: int64

Total Samples: 3653


In [ ]:
df = df.reset_index(drop=True)

print("Index reset successfully.")
print("Total Samples:", len(df))

Index reset successfully.
Total Samples: 3653


In [ ]:
X = df["text"].astype(str)
y = df["category"]

print("Number of texts:", len(X))
print("Number of labels:", len(y))

Number of texts: 3653
Number of labels: 3653


In [ ]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Label Mapping:")

for i, label in enumerate(label_encoder.classes_):
    print(i, "=", label)

Label Mapping:
0 = comp.graphics
1 = rec.sport.hockey
2 = sci.space
3 = talk.politics.misc


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 2922
Testing Samples: 731


In [ ]:
X_train = X_train.tolist()
X_test = X_test.tolist()

y_train = y_train.tolist()
y_test = y_test.tolist()

print("Training data prepared:", len(X_train))
print("Testing data prepared:", len(X_test))

Training data prepared: 2922
Testing data prepared: 731


In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("BERT tokenizer loaded successfully.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT tokenizer loaded successfully.


In [ ]:
def tokenize_text(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
train_encodings = tokenize_text(X_train)

print("Training tokenization completed.")
print("Number of training samples:", len(train_encodings["input_ids"]))

Training tokenization completed.
Number of training samples: 2922


In [ ]:
test_encodings = tokenize_text(X_test)

print("Testing tokenization completed.")
print("Number of testing samples:", len(test_encodings["input_ids"]))

Testing tokenization completed.
Number of testing samples: 731


In [ ]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = NewsDataset(train_encodings, y_train)
test_dataset = NewsDataset(test_encodings, y_test)

print("Train Dataset:", len(train_dataset))
print("Test Dataset:", len(test_dataset))

Train Dataset: 2922
Test Dataset: 731


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
def create_bert_model():
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=4
    )

    model.config.hidden_dropout_prob = 0.1
    model.config.attention_probs_dropout_prob = 0.1

    return model

In [ ]:
def train_bert_model(learning_rate, batch_size, weight_decay, experiment_name):

    model = create_bert_model()

    total_training_steps = int(
        np.ceil(len(train_dataset) / batch_size) * 5
    )

    warmup_steps = int(total_training_steps * 0.1)

    training_args = TrainingArguments(
        output_dir=f"./results/{experiment_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_strategy="epoch",
        report_to="none",
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2
            )
        ]
    )

    trainer.train()

    results = trainer.evaluate()

    predictions = trainer.predict(test_dataset)
    predicted_labels = np.argmax(
        predictions.predictions,
        axis=1
    )

    cm = confusion_matrix(
        y_test,
        predicted_labels
    )

    result = {
        "Experiment": experiment_name,
        "Learning Rate": learning_rate,
        "Batch Size": batch_size,
        "Weight Decay": weight_decay,
        "Accuracy": results["eval_accuracy"],
        "Precision": results["eval_precision"],
        "Recall": results["eval_recall"],
        "F1": results["eval_f1"]
    }

    return trainer, result, cm

In [ ]:
bert_results = []
bert_confusion_matrices = {}
bert_trainers = []

In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00002,
    batch_size=16,
    weight_decay=0.01,
    experiment_name="bert_lr2e-5_bs16_wd001"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.026875,0.331297,5,0.923393,0.923772,0.923393,0.923487


{'Experiment': 'bert_lr2e-5_bs16_wd001', 'Learning Rate': 2e-05, 'Batch Size': 16, 'Weight Decay': 0.01, 'Accuracy': 0.9233926128590971, 'Precision': 0.9237720588157295, 'Recall': 0.9233926128590971, 'F1': 0.923487042567754}

Confusion Matrix:
[[175   5   5   6]
 [  1 184   5   6]
 [  8   3 176   5]
 [  2   3   7 140]]


In [ ]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.15.1


In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00003,
    batch_size=16,
    weight_decay=0.01,
    experiment_name="bert_lr3e-5_bs16_wd001"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.655832,0.261218,0.916553,0.919066,0.916553,0.917148
2,0.180430,0.271563,0.920657,0.921406,0.920657,0.920823
3,0.078638,0.326856,0.927497,0.928466,0.927497,0.927728
4,0.039606,0.386075,0.920657,0.922536,0.920657,0.921085
5,0.020262,0.358830,0.922025,0.923536,0.922025,0.922367


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.020262,0.326784,5,0.927497,0.928466,0.927497,0.927728


{'Experiment': 'bert_lr3e-5_bs16_wd001', 'Learning Rate': 3e-05, 'Batch Size': 16, 'Weight Decay': 0.01, 'Accuracy': 0.9274965800273598, 'Precision': 0.9284663453801026, 'Recall': 0.9274965800273598, 'F1': 0.9277281727249335}

Confusion Matrix:
[[178   5   5   3]
 [  0 181  10   5]
 [  8   0 177   7]
 [  3   1   6 142]]


In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00002,
    batch_size=32,
    weight_decay=0.01,
    experiment_name="bert_lr2e-5_bs32_wd001"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.872181,0.319541,0.906977,0.908748,0.906977,0.907365
2,0.233139,0.249200,0.920657,0.921709,0.920657,0.920832
3,0.116147,0.255013,0.926129,0.929445,0.926129,0.926681
4,0.057427,0.278413,0.919289,0.920248,0.919289,0.919503
5,0.038748,0.276912,0.930233,0.931394,0.930233,0.930433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.038748,0.276912,5,0.930233,0.931394,0.930233,0.930433


{'Experiment': 'bert_lr2e-5_bs32_wd001', 'Learning Rate': 2e-05, 'Batch Size': 32, 'Weight Decay': 0.01, 'Accuracy': 0.9302325581395349, 'Precision': 0.931394394770369, 'Recall': 0.9302325581395349, 'F1': 0.9304329868992917}

Confusion Matrix:
[[175   5   5   6]
 [  0 185   5   6]
 [  6   2 176   8]
 [  3   1   4 144]]


In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00003,
    batch_size=32,
    weight_decay=0.01,
    experiment_name="bert_lr3e-5_bs32_wd001"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.778341,0.286837,0.913817,0.915818,0.913817,0.914245
2,0.196358,0.250121,0.924761,0.926593,0.924761,0.924967
3,0.093698,0.268738,0.923393,0.926799,0.923393,0.924023
4,0.043019,0.270564,0.923393,0.926070,0.923393,0.923846


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.043019,0.249494,4,0.924761,0.926528,0.924761,0.924964


{'Experiment': 'bert_lr3e-5_bs32_wd001', 'Learning Rate': 3e-05, 'Batch Size': 32, 'Weight Decay': 0.01, 'Accuracy': 0.9247606019151847, 'Precision': 0.9265280673679903, 'Recall': 0.9247606019151847, 'F1': 0.9249637236667828}

Confusion Matrix:
[[172   8   6   5]
 [  0 186   4   6]
 [  4   3 175  10]
 [  1   4   4 143]]


In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00002,
    batch_size=16,
    weight_decay=0.1,
    experiment_name="bert_lr2e-5_bs16_wd01"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.757880,0.305711,0.900137,0.902266,0.900137,0.900711
2,0.202254,0.304961,0.915185,0.917234,0.915185,0.915320
3,0.096036,0.328151,0.922025,0.923011,0.922025,0.922185
4,0.052345,0.350514,0.919289,0.921143,0.919289,0.919635
5,0.026667,0.351952,0.920657,0.922726,0.920657,0.921025


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.026667,0.328070,5,0.922025,0.923011,0.922025,0.922185


{'Experiment': 'bert_lr2e-5_bs16_wd01', 'Learning Rate': 2e-05, 'Batch Size': 16, 'Weight Decay': 0.1, 'Accuracy': 0.9220246238030095, 'Precision': 0.9230113018926002, 'Recall': 0.9220246238030095, 'F1': 0.9221852816455035}

Confusion Matrix:
[[175   5   6   5]
 [  2 182   7   5]
 [  9   1 173   9]
 [  2   1   5 144]]


In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00003,
    batch_size=16,
    weight_decay=0.1,
    experiment_name="bert_lr3e-5_bs16_wd01"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.655197,0.260415,0.916553,0.919333,0.916553,0.917173
2,0.179525,0.262704,0.927497,0.927965,0.927497,0.927561
3,0.086541,0.307955,0.930233,0.931960,0.930233,0.930374
4,0.033415,0.370723,0.919289,0.923266,0.919289,0.919895
5,0.017188,0.345661,0.928865,0.930114,0.928865,0.929002


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.017188,0.308276,5,0.930233,0.931960,0.930233,0.930374


{'Experiment': 'bert_lr3e-5_bs16_wd01', 'Learning Rate': 3e-05, 'Batch Size': 16, 'Weight Decay': 0.1, 'Accuracy': 0.9302325581395349, 'Precision': 0.9319602600338456, 'Recall': 0.9302325581395349, 'F1': 0.9303736958776314}

Confusion Matrix:
[[171   5  11   4]
 [  0 184   7   5]
 [  4   4 179   5]
 [  0   2   4 146]]


In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00002,
    batch_size=32,
    weight_decay=0.1,
    experiment_name="bert_lr2e-5_bs32_wd01"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.874401,0.325359,0.900137,0.902563,0.900137,0.900596
2,0.234376,0.245852,0.916553,0.917609,0.916553,0.916732
3,0.116464,0.256316,0.927497,0.929543,0.927497,0.927884
4,0.059128,0.281467,0.922025,0.923141,0.922025,0.922271
5,0.040332,0.278414,0.931601,0.932535,0.931601,0.931797


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.040332,0.278414,5,0.931601,0.932535,0.931601,0.931797


{'Experiment': 'bert_lr2e-5_bs32_wd01', 'Learning Rate': 2e-05, 'Batch Size': 32, 'Weight Decay': 0.1, 'Accuracy': 0.9316005471956225, 'Precision': 0.9325352898130717, 'Recall': 0.9316005471956225, 'F1': 0.9317965071430716}

Confusion Matrix:
[[176   5   4   6]
 [  0 185   5   6]
 [  6   2 177   7]
 [  3   1   5 143]]


In [ ]:
trainer, result, cm = train_bert_model(
    learning_rate=0.00003,
    batch_size=32,
    weight_decay=0.1,
    experiment_name="bert_lr3e-5_bs32_wd01"
)

bert_results.append(result)
bert_confusion_matrices[result["Experiment"]] = cm
bert_trainers.append(trainer)

print(result)
print("\nConfusion Matrix:")
print(cm)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.779696,0.283127,0.919289,0.920493,0.919289,0.919590
2,0.193107,0.234367,0.928865,0.930423,0.928865,0.928959
3,0.086218,0.262596,0.928865,0.931796,0.928865,0.929356
4,0.037781,0.275677,0.928865,0.930658,0.928865,0.929110
5,0.019701,0.272786,0.930233,0.931137,0.930233,0.930410


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.019701,0.272786,5,0.930233,0.931137,0.930233,0.930410


{'Experiment': 'bert_lr3e-5_bs32_wd01', 'Learning Rate': 3e-05, 'Batch Size': 32, 'Weight Decay': 0.1, 'Accuracy': 0.9302325581395349, 'Precision': 0.931136821727847, 'Recall': 0.9302325581395349, 'F1': 0.9304102708515212}

Confusion Matrix:
[[175   6   5   5]
 [  0 185   6   5]
 [  5   1 179   7]
 [  1   4   6 141]]


In [ ]:
bert_results_df = pd.DataFrame(bert_results)

print("BERT Experiment Results:")
display(bert_results_df)

BERT Experiment Results:


,Experiment,Learning Rate,Batch Size,Weight Decay,Accuracy,Precision,Recall,F1
0,bert_lr2e-5_bs16_wd001,0.00002,16,0.01,0.923393,0.923772,0.923393,0.923487
1,bert_lr3e-5_bs16_wd001,0.00003,16,0.01,0.927497,0.928466,0.927497,0.927728
2,bert_lr2e-5_bs32_wd001,0.00002,32,0.01,0.930233,0.931394,0.930233,0.930433
3,bert_lr3e-5_bs32_wd001,0.00003,32,0.01,0.924761,0.926528,0.924761,0.924964
4,bert_lr2e-5_bs16_wd01,0.00002,16,0.10,0.922025,0.923011,0.922025,0.922185
5,bert_lr3e-5_bs16_wd01,0.00003,16,0.10,0.930233,0.931960,0.930233,0.930374
6,bert_lr2e-5_bs32_wd01,0.00002,32,0.10,0.931601,0.932535,0.931601,0.931797
7,bert_lr3e-5_bs32_wd01,0.00003,32,0.10,0.930233,0.931137,0.930233,0.930410


In [ ]:
bert_results_percentage = bert_results_df.copy()

for column in ["Accuracy", "Precision", "Recall", "F1"]:
    bert_results_percentage[column] = (
        bert_results_percentage[column] * 100
    ).round(2).astype(str) + "%"

print("BERT Results in Percentage:")
display(bert_results_percentage)

BERT Results in Percentage:


,Experiment,Learning Rate,Batch Size,Weight Decay,Accuracy,Precision,Recall,F1
0,bert_lr2e-5_bs16_wd001,0.00002,16,0.01,92.34%,92.38%,92.34%,92.35%
1,bert_lr3e-5_bs16_wd001,0.00003,16,0.01,92.75%,92.85%,92.75%,92.77%
2,bert_lr2e-5_bs32_wd001,0.00002,32,0.01,93.02%,93.14%,93.02%,93.04%
3,bert_lr3e-5_bs32_wd001,0.00003,32,0.01,92.48%,92.65%,92.48%,92.5%
4,bert_lr2e-5_bs16_wd01,0.00002,16,0.10,92.2%,92.3%,92.2%,92.22%
5,bert_lr3e-5_bs16_wd01,0.00003,16,0.10,93.02%,93.2%,93.02%,93.04%
6,bert_lr2e-5_bs32_wd01,0.00002,32,0.10,93.16%,93.25%,93.16%,93.18%
7,bert_lr3e-5_bs32_wd01,0.00003,32,0.10,93.02%,93.11%,93.02%,93.04%


In [ ]:
best_bert_index = bert_results_df["F1"].idxmax()

best_bert_result = bert_results_df.loc[best_bert_index]

print("Best BERT Combination:")
print(best_bert_result)

Best BERT Combination:
Experiment       bert_lr2e-5_bs32_wd01
Learning Rate                  0.00002
Batch Size                          32
Weight Decay                       0.1
Accuracy                      0.931601
Precision                     0.932535
Recall                        0.931601
F1                            0.931797
Name: 6, dtype: object


In [ ]:
print("Best BERT Performance")
print("---------------------")

print("Learning Rate :", best_bert_result["Learning Rate"])
print("Batch Size    :", best_bert_result["Batch Size"])
print("Weight Decay  :", best_bert_result["Weight Decay"])
print("Accuracy      :", round(best_bert_result["Accuracy"] * 100, 2), "%")
print("Precision     :", round(best_bert_result["Precision"] * 100, 2), "%")
print("Recall        :", round(best_bert_result["Recall"] * 100, 2), "%")
print("F1-score      :", round(best_bert_result["F1"] * 100, 2), "%")

Best BERT Performance
---------------------
Learning Rate : 2e-05
Batch Size    : 32
Weight Decay  : 0.1
Accuracy      : 93.16 %
Precision     : 93.25 %
Recall        : 93.16 %
F1-score      : 93.18 %


In [ ]:
best_experiment_name = best_bert_result["Experiment"]

best_cm = bert_confusion_matrices[best_experiment_name]

print("Best BERT Confusion Matrix:")
print(best_cm)

Best BERT Confusion Matrix:
[[176   5   4   6]
 [  0 185   5   6]
 [  6   2 177   7]
 [  3   1   5 143]]


In [ ]:
best_trainer = bert_trainers[best_bert_index]

best_predictions = best_trainer.predict(test_dataset)

best_predicted_labels = np.argmax(
    best_predictions.predictions,
    axis=1
)

print("Classification Report:")
print(
    classification_report(
        y_test,
        best_predicted_labels,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Classification Report:
                    precision    recall  f1-score   support

     comp.graphics       0.95      0.92      0.94       191
  rec.sport.hockey       0.96      0.94      0.95       196
         sci.space       0.93      0.92      0.92       192
talk.politics.misc       0.88      0.94      0.91       152

          accuracy                           0.93       731
         macro avg       0.93      0.93      0.93       731
      weighted avg       0.93      0.93      0.93       731



In [ ]:
best_bert_test_predictions = best_predicted_labels.copy()

print("Best BERT test predictions saved.")
print("Number of predictions:", len(best_bert_test_predictions))

Best BERT test predictions saved.
Number of predictions: 731


In [ ]:
prediction_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": best_bert_test_predictions
})

prediction_df.to_csv(
    "dataset2_best_bert_predictions.csv",
    index=False
)

print("Prediction file saved successfully.")

Prediction file saved successfully.


In [ ]:
import torch.nn.functional as F

best_probs = F.softmax(torch.tensor(best_predictions.predictions), dim=1).numpy()

bert_prediction_df = pd.DataFrame(
    best_probs,
    columns=[f"prob_{label}" for label in label_encoder.classes_]
)

bert_prediction_df["true_label"] = y_test
bert_prediction_df["predicted_label"] = best_bert_test_predictions

bert_prediction_df.to_csv("dataset2_best_bert_predictions.csv", index=False)

print("Best BERT probabilities + predictions saved successfully.")

Best BERT probabilities + predictions saved successfully.


In [ ]:
bert_true = np.array(y_test)


DISTIL_PRED_PATH = "/content/best_distilbert_test_predictions.csv"
distil_df = pd.read_csv(DISTIL_PRED_PATH)

distil_prob_cols = [c for c in distil_df.columns if c.startswith("prob_")]
distil_probs = distil_df[distil_prob_cols].values
distil_true = distil_df["true_label"].values


assert best_probs.shape == distil_probs.shape, "Prob array shapes don't match!"
assert np.array_equal(bert_true, distil_true), \
    "True labels don't match between BERT and DistilBERT test sets — check split/order!"

print("Test sets aligned correctly. Samples:", len(bert_true))


ensemble_probs = (best_probs + distil_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)
ensemble_true = bert_true


acc = accuracy_score(ensemble_true, ensemble_preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    ensemble_true, ensemble_preds, average="macro", zero_division=0
)

print("\nENSEMBLE RESULTS (BERT + DistilBERT) — Dataset-2")
print("=" * 50)
print("Accuracy :", round(acc * 100, 2), "%")
print("Precision:", round(prec * 100, 2), "%")
print("Recall   :", round(rec * 100, 2), "%")
print("F1       :", round(f1 * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    ensemble_true, ensemble_preds,
    target_names=label_encoder.classes_,
    digits=4, zero_division=0
))

cm = confusion_matrix(ensemble_true, ensemble_preds)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print("Confusion Matrix:")
display(cm_df)


ensemble_row = pd.DataFrame([{
    "Model": "ENSEMBLE (BERT + DistilBERT)",
    "Dataset": "Dataset-2",
    "Acc": acc * 100,
    "Prec": prec * 100,
    "Rec": rec * 100,
    "F1": f1 * 100
}])
display(ensemble_row)

ensemble_row.to_csv("dataset2_ensemble_final_results.csv", index=False)
print("\nSaved ensemble result CSV.")

Test sets aligned correctly. Samples: 731

ENSEMBLE RESULTS (BERT + DistilBERT) — Dataset-2
Accuracy : 93.16 %
Precision: 93.05 %
Recall   : 93.16 %
F1       : 93.09 %

Classification Report:
                    precision    recall  f1-score   support

     comp.graphics     0.9514    0.9215    0.9362       191
  rec.sport.hockey     0.9442    0.9490    0.9466       196
         sci.space     0.9219    0.9219    0.9219       192
talk.politics.misc     0.9045    0.9342    0.9191       152

          accuracy                         0.9316       731
         macro avg     0.9305    0.9316    0.9309       731
      weighted avg     0.9319    0.9316    0.9317       731

Confusion Matrix:


,comp.graphics,rec.sport.hockey,sci.space,talk.politics.misc
comp.graphics,176,6,5,4
rec.sport.hockey,0,186,4,6
sci.space,7,3,177,5
talk.politics.misc,2,2,6,142


,Model,Dataset,Acc,Prec,Rec,F1
0,ENSEMBLE (BERT + DistilBERT),Dataset-2,93.160055,93.046185,93.163277,93.092599



Saved ensemble result CSV.


In [ ]:

bert_row = {
    "Model": "BERT",
    "Learning Rate": best_bert_result["Learning Rate"],
    "Batch Size": best_bert_result["Batch Size"],
    "Weight Decay": best_bert_result["Weight Decay"],
    "Acc": round(best_bert_result["Accuracy"] * 100, 2),
    "Prec": round(best_bert_result["Precision"] * 100, 2),
    "Rec": round(best_bert_result["Recall"] * 100, 2),
    "F1": round(best_bert_result["F1"] * 100, 2)
}


distilbert_row = {
    "Model": "BERT Variation (DistilBERT)",
    "Learning Rate": 0.00002,
    "Batch Size": 16,
    "Weight Decay": 0.01,
    "Acc": round(92.20246238030096, 2),
    "Prec": round(92.15964520533049, 2),
    "Rec": round(92.15031665738766, 2),
    "F1": round(92.13608041305318, 2)
}

ensemble_summary_row = {
    "Model": "ENSEMBLE",
    "Learning Rate": "-",
    "Batch Size": "-",
    "Weight Decay": "-",
    "Acc": round(acc * 100, 2),
    "Prec": round(prec * 100, 2),
    "Rec": round(rec * 100, 2),
    "F1": round(f1 * 100, 2)
}

dataset2_summary_df = pd.DataFrame([bert_row, distilbert_row, ensemble_summary_row])

print("DATASET-2 FINAL SUMMARY TABLE")
print("=" * 70)
display(dataset2_summary_df)

dataset2_summary_df.to_csv("dataset2_final_summary_table.csv", index=False)
print("\nSaved: dataset2_final_summary_table.csv")

DATASET-2 FINAL SUMMARY TABLE


,Model,Learning Rate,Batch Size,Weight Decay,Acc,Prec,Rec,F1
0,BERT,0.00002,32,0.1,93.16,93.25,93.16,93.18
1,BERT Variation (DistilBERT),0.00002,16,0.01,92.20,92.16,92.15,92.14
2,ENSEMBLE,-,-,-,93.16,93.05,93.16,93.09



Saved: dataset2_final_summary_table.csv
